# 05 — One contract across healthcare and controls

After domain-specific mapping, the same directional equations, boundary
operators, and occurrence schema can be used. Scientific meaning does not
transfer automatically.


In [ ]:
import numpy as np
import pandas as pd
import featuregraph as fg
from featuregraph.operators.states import rising_state

respiration = pd.DataFrame({
    "domain": "healthcare", "series_id": "respiration-01",
    "sample": np.arange(10),
    "signal": [0, .5, 1, 1, .5, 0, 0, .6, 1.1, .5],
    "unit": "normalized respiration",
})
pressure = pd.DataFrame({
    "domain": "controls", "series_id": "pressure-01",
    "sample": np.arange(10),
    "signal": [2800, 2801, 2803, 2803, 2802, 2801, 2801, 2802, 2804, 2802],
    "unit": "pressure-like value",
})
observations = pd.concat([respiration, pressure], ignore_index=True)
observations["rate"] = observations.groupby("series_id")["signal"].diff().fillna(0)


In [ ]:
rate = {"column": "rate"}; eps = {"parameter": "eps"}
contract = {
    "version": "state-contract-v1", "parameters": {"eps": 0.0},
    "group_by": "series_id",
    "states": {
        "rising": {"op": "gt", "left": rate, "right": eps},
        "falling": {"op": "lt", "left": rate,
                    "right": {"op": "neg", "value": eps}},
        "inactive": {"op": "le", "left": {"op": "abs", "value": rate},
                     "right": eps},
    },
    "events": {"enter_state": {"type": "enter_label"},
               "exit_state": {"type": "exit_label"}},
}
compiled = fg.compile_states(observations, contract)


In [ ]:
objects = compiled.observations.groupby(
    ["domain", "series_id", "state_occurrence_id", "state"], sort=False
).agg(
    start_sample=("sample", "min"), end_sample=("sample", "max"),
    sample_count=("sample", "size"), start_value=("signal", "first"),
    end_value=("signal", "last"), unit=("unit", "first"),
).reset_index()
objects["net_change"] = objects["end_value"] - objects["start_value"]
objects


In [ ]:
columns = [
    "state_occurrence_id", "state", "start_sample", "end_sample",
    "sample_count", "start_value", "end_value", "unit", "net_change",
]
schema = objects.groupby("domain", sort=False).apply(
    lambda x: tuple(x[columns].columns), include_groups=False
)
objects.groupby(["domain", "state"]).size().unstack(fill_value=0)


In [ ]:
respiration_check = fg.transition.Transition(
    respiration.copy(), "signal", "rising", rising_state, eps=0
)
pressure_check = fg.transition.Transition(
    pressure.copy(), "signal", "rising", rising_state, eps=0
)
assert compiled.validation_report["passed"].all()
assert schema.loc["healthcare"] == schema.loc["controls"]
assert set(objects["state"]) == {"rising", "falling", "inactive"}
assert respiration_check.df["signal_rising"].any()
assert pressure_check.df["signal_rising"].any()


Transferred: canonical columns, state equations, events,
identity, and schema. Not transferred: units, preprocessing justification, or
claims about breaths, stress, faults, causes, or prediction. FeatureGraph
supplies structural and analytical understanding; scientific understanding
remains domain-specific.
